# Milestone 1 — Baseline Text-to-Image Runs
IE 7615 Generative Project (Group 10)

Pretrained Stable Diffusion 1.5 baseline. Runs 5 prompts of increasing complexity,
saves images + a reproducibility log + a gallery index, then zips them for download.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

## 1. Install

In [ ]:
# Colab: set runtime to T4 GPU first.
!pip -q install "diffusers==0.31.0" "transformers==4.46.3" "accelerate==1.1.1" safetensors

## 2. Imports & environment check

In [ ]:
import os, json, platform, datetime
import torch
from diffusers import StableDiffusionPipeline

import diffusers, transformers
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(f"torch {torch.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")

## 3. Config

In [ ]:
MODEL_ID   = "stable-diffusion-v1-5/stable-diffusion-v1-5"  # fallback: "sd-legacy/stable-diffusion-v1-5"
SEED       = 42
STEPS      = 30
GUIDANCE   = 7.5           # classifier-free guidance scale (baseline default)
HEIGHT     = 512
WIDTH      = 512
NEG_PROMPT = "blurry, lowres, distorted, watermark, text"

OUT_DIR = "outputs/milestone1_baseline"
IMG_DIR = os.path.join(OUT_DIR, "images")
os.makedirs(IMG_DIR, exist_ok=True)

# 5 prompts chosen to span complexity (single object -> abstract)
PROMPTS = [
    ("p1_simple_object",   "a red apple on a wooden table"),
    ("p2_multi_object",    "a cat and a dog sitting together on a park bench"),
    ("p3_spatial",         "a blue cube to the left of a yellow sphere on a white floor"),
    ("p4_stylistic",       "a mountain lake at sunrise, in the style of a watercolor painting"),
    ("p5_abstract",        "the feeling of nostalgia, surreal dreamlike digital art"),
]

## 4. Load pipeline

In [ ]:
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
pipe.enable_attention_slicing()   # keeps memory low on a T4
# Safety checker stays ON by default. If a benign prompt returns a black image
# (false positive), you can disable it: pipe.safety_checker = None
scheduler_name = type(pipe.scheduler).__name__
print("Scheduler:", scheduler_name)

## 5. Generate images + collect run metadata

In [ ]:
device = pipe.device.type
runs = []

for tag, prompt in PROMPTS:
    generator = torch.Generator(device=device).manual_seed(SEED)  # per-prompt reproducibility
    result = pipe(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        height=HEIGHT,
        width=WIDTH,
        generator=generator,
    )
    image = result.images[0]
    fname = f"{tag}.png"
    image.save(os.path.join(IMG_DIR, fname))

    runs.append({
        "tag": tag,
        "prompt": prompt,
        "negative_prompt": NEG_PROMPT,
        "seed": SEED,
        "steps": STEPS,
        "guidance_scale": GUIDANCE,
        "height": HEIGHT,
        "width": WIDTH,
        "image": f"images/{fname}",
    })
    print(f"[done] {tag}: {prompt[:50]}")

## 6. Write reproducibility log + gallery index

In [ ]:
run_log = {
    "generated_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "model_id": MODEL_ID,
    "scheduler": scheduler_name,
    "device": device,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "dtype": str(dtype),
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "diffusers": diffusers.__version__,
        "transformers": transformers.__version__,
    },
    "num_runs": len(runs),
    "runs": runs,
}
with open(os.path.join(OUT_DIR, "run_log.json"), "w") as f:
    json.dump(run_log, f, indent=2)

with open(os.path.join(OUT_DIR, "GALLERY.md"), "w") as f:
    f.write("# Milestone 1 - Baseline prompt->image runs\n\n")
    f.write(f"Model: `{MODEL_ID}` | Scheduler: {scheduler_name} | "
            f"Seed: {SEED} | Steps: {STEPS} | Guidance: {GUIDANCE}\n\n")
    for r in runs:
        f.write(f"### {r['tag']}\n")
        f.write(f"**Prompt:** {r['prompt']}\n\n")
        f.write(f"![{r['tag']}]({r['image']})\n\n")

print("Saved:", OUT_DIR)

## 7. Preview the 5 images

In [ ]:
from IPython.display import Image, display
for r in runs:
    print(r["prompt"])
    display(Image(filename=os.path.join(OUT_DIR, r["image"])))

## 8. Download everything as a zip\nUnzip into your local repo's `outputs/` folder, then commit + push.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("milestone1_baseline", "zip", "outputs/milestone1_baseline")
files.download("milestone1_baseline.zip")